# Engenharia de features — Cadastros suspeitos

*Responsáveis:* Emilly Siqueira e Cleber Campos

## Objetivo
Preparar os atributos de clientes e endereços para um primeiro
modelo de identificação de cadastros que precisam de revisão.

## Fontes
- Silver: ecommerce_clientes
- Silver: ecommerce_enderecos

## Atributos candidatos
- Domínio do e-mail.
- Tamanho do usuário do e-mail.
- Quantidade de dígitos no e-mail.
- Tamanho do hash.
- Idade do cadastro na referência.
- Quantidade de endereços disponíveis na referência.
- Indicador de endereço disponível.
- Proporção de logradouros ausentes.
- Proporções de coordenadas ausentes e inválidas,
  sujeitas à avaliação de utilidade.

## Critérios
- Ler os dados diretamente da Silver.
- Executar sem depender de variáveis do notebook de EDA.
- Considerar somente endereços ingeridos até a referência do cliente.
- Preservar a diferença entre informação ausente e valor zero.
- Conferir o rótulo experimental antes do treinamento.
- Não alterar as tabelas de origem.

In [0]:
# ============================================================
# BIBLIOTECAS — ENGENHARIA DE FEATURES
# ============================================================

import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

print("Bibliotecas carregadas.")

In [0]:
# ============================================================
# AUTENTICAÇÃO ADLS — CAMINHO COMPLETO DO .ENV
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

caminho_env = Path(
    "/Workspace/Users/cleber7engmecanica@gmail.com/"
    "estagio-empregadados-turma-2/.env"
)

if not caminho_env.is_file():
    raise FileNotFoundError(
        "O arquivo .env não está acessível neste caminho. "
        "Confira o caminho exibido no notebook antigo."
    )

load_dotenv(dotenv_path=caminho_env, override=True)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

configuracoes = {
    "ADLS_CLIENT_ID": client_id,
    "ADLS_TENANT_ID": tenant_id,
    "ADLS_CLIENT_SECRET": client_secret,
    "ADLS_STORAGE_ACCOUNT_NAME": storage_account,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if not valor or not valor.strip()
]

if faltantes:
    raise ValueError(
        "Configurações ausentes ou vazias no .env: "
        + ", ".join(faltantes)
    )

host_adls = f"{storage_account}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{host_adls}":
        "OAuth",

    f"fs.azure.account.oauth.provider.type.{host_adls}":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{host_adls}":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{host_adls}":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{host_adls}":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

print("As quatro configurações foram carregadas.")
print("Opções do ADLS preparadas.")
print("O acesso será confirmado na leitura das tabelas Silver.")

In [0]:
# ============================================================
# LEITURA DAS FONTES — CLIENTES E ENDEREÇOS
# ============================================================

base_silver = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/silver"
)

clientes_path = f"{base_silver}/ecommerce_clientes"
enderecos_path = f"{base_silver}/ecommerce_enderecos"

df_clientes_ml = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(clientes_path)
)

df_enderecos_ml = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(enderecos_path)
)

for nome, df in [
    ("CLIENTES", df_clientes_ml),
    ("ENDEREÇOS", df_enderecos_ml),
]:
    print(f"\n===== {nome} =====")
    print("Quantidade de registros:", df.count())
    print("Quantidade de colunas:", len(df.columns))
    df.printSchema()

print("\nLeitura concluída. Nenhum dado da Silver foi alterado.")

In [0]:
# ============================================================
# ATRIBUTOS DOS CLIENTES
# ============================================================

from pyspark.sql import functions as F

# 1. Validar as colunas necessárias
colunas_necessarias = [
    "id_cliente",
    "email",
    "senha_hash",
    "dt_cadastro",
    "bronze_ingested_at",
    "bronze_source_file",
]

faltantes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in df_clientes_ml.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes na base de clientes: {faltantes}"
    )

# Os cálculos abaixo pressupõem datas já convertidas na Silver.
tipos = dict(df_clientes_ml.dtypes)

for coluna in ["dt_cadastro", "bronze_ingested_at"]:
    if tipos[coluna] != "timestamp":
        raise ValueError(
            f"{coluna} deveria ser timestamp, "
            f"mas está como {tipos[coluna]}. "
            "Confira o tipo antes de calcular a idade."
        )

# 2. Criar os atributos com os mesmos critérios da EDA
regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df_features_clientes = (
    df_clientes_ml
    .withColumn(
        "_email_formato_valido",
        F.coalesce(
            F.col("email").rlike(regex_email),
            F.lit(False)
        )
    )
    .withColumn(
        "_usuario_email",
        F.when(
            F.col("_email_formato_valido"),
            F.regexp_extract("email", r"^([^@]+)@", 1)
        )
    )
    .withColumn(
        "dominio_email",
        F.when(
            F.col("_email_formato_valido"),
            F.lower(
                F.regexp_extract("email", r"@([^@]+)$", 1)
            )
        )
    )
    .withColumn(
        "tamanho_usuario_email",
        F.length("_usuario_email")
    )
    # Conta dígitos somente na parte anterior ao @.
    .withColumn(
        "qtd_digitos_email",
        F.length(
            F.regexp_replace("_usuario_email", r"[^0-9]", "")
        )
    )
    .withColumn(
        "tamanho_hash",
        F.length("senha_hash")
    )
    # Diferença em dias, incluindo frações.
    # Valores negativos são preservados.
    .withColumn(
        "idade_cadastro_dias",
        (
            F.col("bronze_ingested_at").cast("double")
            - F.col("dt_cadastro").cast("double")
        ) / F.lit(86400.0)
    )
    .select(
        # Identificação e rastreabilidade.
        "id_cliente",
        "bronze_source_file",
        "bronze_ingested_at",

        # Atributos candidatos.
        "dominio_email",
        "tamanho_usuario_email",
        "qtd_digitos_email",
        "tamanho_hash",
        "idade_cadastro_dias",
    )
)

# 3. Validar contagens e valores ausentes
atributos_clientes = [
    "dominio_email",
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
]

resumo = df_features_clientes.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_cliente").alias("clientes_distintos"),

    *[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(f"{coluna}_nulos")
        for coluna in [
            "id_cliente",
            "bronze_ingested_at",
            *atributos_clientes,
        ]
    ],

    F.count(
        F.when(F.col("idade_cadastro_dias") < 0, 1)
    ).alias("idades_negativas")
).first().asDict()

total_origem = df_clientes_ml.count()

if resumo["total_registros"] != total_origem:
    raise ValueError(
        "A quantidade de linhas mudou na criação dos atributos."
    )

if (
    resumo["id_cliente_nulos"] > 0
    or resumo["bronze_ingested_at_nulos"] > 0
):
    raise ValueError(
        "Existem clientes sem ID ou sem instante de ingestão. "
        "Revisar antes da junção temporal com endereços."
    )

print("===== ATRIBUTOS DOS CLIENTES =====")
print("Linhas na origem:", total_origem)

for indicador, valor in resumo.items():
    print(f"{indicador}: {valor}")

print("\nAtributos criados:")
for atributo in atributos_clientes:
    print("-", atributo)

print("\nIdade negativa indica cadastro posterior à ingestão.")
print("As cópias ainda serão consolidadas em uma etapa posterior.")
print("Nenhum rótulo foi criado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# REFERÊNCIAS TEMPORAIS DOS CLIENTES
# ============================================================

from pyspark.sql import functions as F

# 1. Validar as colunas necessárias
colunas_necessarias = [
    "id_cliente",
    "bronze_ingested_at",
]

faltantes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in df_features_clientes.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes na base de atributos: {faltantes}"
    )

# 2. Verificar se as referências estão preenchidas
referencias_nulas = (
    df_features_clientes
    .filter(
        F.col("id_cliente").isNull()
        | F.col("bronze_ingested_at").isNull()
    )
    .count()
)

if referencias_nulas > 0:
    raise ValueError(
        f"Existem {referencias_nulas} linhas sem ID de cliente "
        "ou sem instante de ingestão."
    )

# 3. Criar uma linha por cliente e instante de referência
# A base de atributos original continua preservada.
df_referencias_clientes = (
    df_features_clientes
    .select(
        "id_cliente",
        F.col("bronze_ingested_at").alias("instante_referencia")
    )
    .distinct()
)

# 4. Resumir as referências
resumo_referencias = (
    df_referencias_clientes
    .agg(
        F.count("*").alias("total_referencias"),
        F.countDistinct("id_cliente").alias("clientes_distintos"),
        F.min("instante_referencia").alias("primeira_referencia"),
        F.max("instante_referencia").alias("ultima_referencia"),
    )
    .first()
    .asDict()
)

if resumo_referencias["total_referencias"] == 0:
    raise ValueError("Não existem referências para processar.")

clientes_com_varias_referencias = (
    df_referencias_clientes
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("===== REFERÊNCIAS TEMPORAIS =====")
print("Linhas de clientes:", df_features_clientes.count())

for indicador, valor in resumo_referencias.items():
    print(f"{indicador}: {valor}")

print(
    "Clientes com mais de uma referência:",
    clientes_com_varias_referencias
)

print("\nReferência utilizada: ingestão do registro de cliente.")
print("Os endereços serão filtrados por essa referência.")
print("A base de atributos de clientes permanece preservada.")
print("Nenhum dado foi gravado.")

In [0]:
# ============================================================
# ATRIBUTOS DOS ENDEREÇOS POR REFERÊNCIA
# ============================================================

from pyspark.sql import functions as F

# 1. Validar as colunas necessárias
necessarias = [
    "id_cliente",
    "id_endereco",
    "bronze_ingested_at",
    "logradouro",
    "latitude",
    "longitude",
]

faltantes = [
    coluna
    for coluna in necessarias
    if coluna not in df_enderecos_ml.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes em endereços: {faltantes}"
    )

if dict(df_enderecos_ml.dtypes)["bronze_ingested_at"] != "timestamp":
    raise ValueError(
        "bronze_ingested_at dos endereços deve ser timestamp."
    )

# A agregação pressupõe uma linha por endereço.
validacao_ids = df_enderecos_ml.agg(
    F.count("*").alias("linhas"),
    F.count("id_endereco").alias("ids_preenchidos"),
    F.countDistinct("id_endereco").alias("ids_distintos"),
).first()

if not (
    validacao_ids["linhas"]
    == validacao_ids["ids_preenchidos"]
    == validacao_ids["ids_distintos"]
):
    raise ValueError(
        "Existem IDs de endereço nulos ou repetidos. "
        "Revisar antes de calcular os atributos."
    )

# 2. Preparar os campos sem alterar a Silver
df_enderecos_preparados = (
    df_enderecos_ml
    .select(
        "id_cliente",
        "id_endereco",
        F.col("bronze_ingested_at").alias("ingestao_endereco"),
        F.trim(F.col("logradouro").cast("string")).alias(
            "_logradouro_texto"
        ),
        F.trim(F.col("latitude").cast("string")).alias(
            "_latitude_texto"
        ),
        F.trim(F.col("longitude").cast("string")).alias(
            "_longitude_texto"
        ),
    )
    .withColumn(
        "_latitude_num",
        F.expr("try_cast(_latitude_texto AS DOUBLE)")
    )
    .withColumn(
        "_longitude_num",
        F.expr("try_cast(_longitude_texto AS DOUBLE)")
    )
)

logradouro_ausente = (
    F.col("_logradouro_texto").isNull()
    | (F.col("_logradouro_texto") == "")
)

latitude_ausente = (
    F.col("_latitude_texto").isNull()
    | (F.col("_latitude_texto") == "")
)

longitude_ausente = (
    F.col("_longitude_texto").isNull()
    | (F.col("_longitude_texto") == "")
)

# Valor preenchido, mas não numérico, NaN ou fora da faixa.
latitude_invalida = (
    (~latitude_ausente)
    & (
        F.col("_latitude_num").isNull()
        | F.isnan("_latitude_num")
        | (~F.col("_latitude_num").between(-90.0, 90.0))
    )
)

longitude_invalida = (
    (~longitude_ausente)
    & (
        F.col("_longitude_num").isNull()
        | F.isnan("_longitude_num")
        | (~F.col("_longitude_num").between(-180.0, 180.0))
    )
)

df_enderecos_atributos = (
    df_enderecos_preparados
    .select(
        "id_cliente",
        "id_endereco",
        "ingestao_endereco",
        logradouro_ausente.cast("int").alias(
            "_logradouro_ausente"
        ),
        (latitude_ausente | longitude_ausente)
        .cast("int")
        .alias("_coordenadas_ausentes"),
        (latitude_invalida | longitude_invalida)
        .cast("int")
        .alias("_coordenadas_invalidas"),
    )
)

# 3. Selecionar endereços disponíveis até a referência
# Sem correspondência, o left join preserva a referência.
df_enderecos_no_instante = (
    df_referencias_clientes.alias("c")
    .join(
        df_enderecos_atributos.alias("e"),
        (
            F.col("c.id_cliente") == F.col("e.id_cliente")
        )
        & (
            F.col("e.ingestao_endereco")
            <= F.col("c.instante_referencia")
        ),
        "left",
    )
    .select(
        F.col("c.id_cliente").alias("id_cliente"),
        F.col("c.instante_referencia"),
        F.col("e.id_endereco"),
        F.col("e._logradouro_ausente"),
        F.col("e._coordenadas_ausentes"),
        F.col("e._coordenadas_invalidas"),
    )
)

# 4. Agregar por cliente e referência
# Sem endereço disponível: quantidade zero e proporções nulas.
df_features_enderecos = (
    df_enderecos_no_instante
    .groupBy("id_cliente", "instante_referencia")
    .agg(
        F.count("id_endereco").alias(
            "qtd_enderecos_disponiveis"
        ),
        F.avg("_logradouro_ausente").alias(
            "proporcao_logradouros_ausentes"
        ),
        F.avg("_coordenadas_ausentes").alias(
            "proporcao_coordenadas_ausentes"
        ),
        F.avg("_coordenadas_invalidas").alias(
            "proporcao_coordenadas_invalidas"
        ),
    )
    .withColumn(
        "tem_endereco_disponivel",
        F.col("qtd_enderecos_disponiveis") > 0
    )
)

# 5. Validar a preservação das referências
total_referencias = df_referencias_clientes.count()
total_resultado = df_features_enderecos.count()

if total_resultado != total_referencias:
    raise ValueError(
        "A quantidade de referências mudou durante a agregação."
    )

proporcoes = [
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas",
]

inconsistencia = F.lit(False)

for coluna in proporcoes:
    valor = F.col(coluna)

    inconsistencia = inconsistencia | (
        F.col("tem_endereco_disponivel")
        & (
            valor.isNull()
            | F.isnan(valor)
            | (~valor.between(0.0, 1.0))
        )
    ) | (
        (~F.col("tem_endereco_disponivel"))
        & valor.isNotNull()
    )

if df_features_enderecos.filter(inconsistencia).limit(1).count() > 0:
    raise ValueError(
        "Foram encontradas proporções inconsistentes."
    )

# 6. Exibir o resumo
print("===== ATRIBUTOS DOS ENDEREÇOS =====")
print("Referências de origem:", total_referencias)
print("Referências após agregação:", total_resultado)

display(
    df_features_enderecos
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_referencias"),
        F.sum("qtd_enderecos_disponiveis").alias(
            "total_enderecos_disponiveis"
        ),
    )
    .orderBy("tem_endereco_disponivel")
)

resumo_nulos = df_features_enderecos.agg(
    *[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in proporcoes
    ]
).first().asDict()

print("\n===== NULOS NAS PROPORÇÕES =====")

for coluna, quantidade in resumo_nulos.items():
    print(f"{coluna}: {quantidade}")

print("\nSem endereço disponível, as proporções permanecem nulas.")
print("Proporção zero indica ausência do problema nos endereços avaliados.")
print("Coordenada zero, isoladamente, não é considerada inválida.")
print("Faixas válidas não comprovam a localização correta do endereço.")
print("Nenhum rótulo foi criado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# UNIÃO DOS ATRIBUTOS DE CLIENTES E ENDEREÇOS
# ============================================================

from pyspark.sql import functions as F

chaves = ["id_cliente", "bronze_ingested_at"]

atributos_clientes = [
    "dominio_email",
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
]

atributos_enderecos = [
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas",
]

# 1. Validar as colunas necessárias
for nome, df, necessarias in [
    (
        "clientes",
        df_features_clientes,
        chaves + atributos_clientes,
    ),
    (
        "endereços",
        df_features_enderecos,
        ["id_cliente", "instante_referencia"]
        + atributos_enderecos,
    ),
]:
    faltantes = [
        coluna
        for coluna in necessarias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

conflitos = [
    coluna
    for coluna in atributos_enderecos
    if coluna in df_features_clientes.columns
]

if conflitos:
    raise ValueError(
        f"A base de clientes já contém atributos de endereços: {conflitos}"
    )

# 2. Preparar a base agregada de endereços para a junção
df_enderecos_para_join = (
    df_features_enderecos
    .select(
        "id_cliente",
        F.col("instante_referencia").alias("bronze_ingested_at"),
        *atributos_enderecos,
    )
    .withColumn("_referencia_encontrada", F.lit(True))
)

# Uma linha por chave evita multiplicar os registros de clientes.
chaves_repetidas = (
    df_enderecos_para_join
    .groupBy(*chaves)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if chaves_repetidas > 0:
    raise ValueError(
        "Existem referências repetidas na agregação de endereços."
    )

# 3. Unir preservando todas as linhas e colunas de clientes
df_features_unidas = (
    df_features_clientes
    .join(
        df_enderecos_para_join,
        on=chaves,
        how="left",
    )
)

# 4. Validar a junção
total_antes = df_features_clientes.count()

validacao = df_features_unidas.agg(
    F.count("*").alias("total_depois"),
    F.countDistinct("id_cliente").alias("clientes_distintos"),
    F.count(
        F.when(F.col("_referencia_encontrada").isNull(), 1)
    ).alias("linhas_sem_correspondencia"),
).first()

if validacao["total_depois"] != total_antes:
    raise ValueError(
        "A quantidade de linhas mudou durante a junção."
    )

if validacao["linhas_sem_correspondencia"] > 0:
    raise ValueError(
        "Há referências de clientes ausentes na agregação "
        "de endereços. Revise a execução das células anteriores."
    )

df_features_unidas = df_features_unidas.drop(
    "_referencia_encontrada"
)

# 5. Exibir os resultados
print("===== VALIDAÇÃO DA UNIÃO =====")
print("Linhas antes:", total_antes)
print("Linhas depois:", validacao["total_depois"])
print("Clientes distintos:", validacao["clientes_distintos"])
print(
    "Linhas sem correspondência na agregação:",
    validacao["linhas_sem_correspondencia"],
)

print("\n===== DISPONIBILIDADE NA BASE UNIDA =====")

display(
    df_features_unidas
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.countDistinct("id_cliente").alias(
            "qtd_clientes_distintos"
        ),
    )
    .orderBy("tem_endereco_disponivel")
)

print("\n===== NULOS POR ATRIBUTO =====")

todos_atributos = atributos_clientes + atributos_enderecos

resumo_nulos = df_features_unidas.agg(
    *[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in todos_atributos
    ]
).first().asDict()

for coluna, quantidade in resumo_nulos.items():
    print(f"{coluna}: {quantidade}")

print("\n===== ATRIBUTOS PRESERVADOS =====")

for coluna in todos_atributos:
    print("-", coluna)

print("\nAs linhas repetidas ainda estão presentes.")
print("As proporções sem endereço disponível permanecem nulas.")
print("Domínio, padrão do e-mail, hash e idade foram preservados.")
print("Nenhum rótulo foi criado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# CONSOLIDAÇÃO DE CÓPIAS IDÊNTICAS NA BASE DE ATRIBUTOS
# ============================================================

from pyspark.sql import functions as F

# 1. Consolidar somente linhas iguais em TODAS as colunas
# Inclui ID, referência temporal, arquivo de origem e atributos.
# Não escolhe arbitrariamente um registro por cliente.
df_features_sem_copias = df_features_unidas.distinct()

# 2. Calcular as contagens
total_antes = df_features_unidas.count()

resumo = df_features_sem_copias.agg(
    F.count("*").alias("total_depois"),
    F.countDistinct("id_cliente").alias("clientes_distintos"),
    F.count(
        F.when(F.col("id_cliente").isNull(), 1)
    ).alias("ids_nulos"),
).first().asDict()

df_clientes_multiplas_linhas = (
    df_features_sem_copias
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
)

clientes_multiplas_linhas = df_clientes_multiplas_linhas.count()

print("===== CONSOLIDAÇÃO DAS CÓPIAS =====")
print("Linhas antes:", total_antes)
print("Linhas após consolidação:", resumo["total_depois"])
print(
    "Cópias excedentes:",
    total_antes - resumo["total_depois"]
)
print("Clientes distintos:", resumo["clientes_distintos"])
print("IDs nulos:", resumo["ids_nulos"])
print(
    "Clientes ainda com múltiplas linhas:",
    clientes_multiplas_linhas
)

# 3. Validar a granularidade desta versão
if resumo["total_depois"] == 0:
    raise ValueError("A base de atributos está vazia.")

if resumo["ids_nulos"] > 0:
    raise ValueError(
        "Existem IDs de cliente nulos. Revisar antes de continuar."
    )

if clientes_multiplas_linhas > 0:
    raise ValueError(
        "Ainda existem clientes com registros diferentes. "
        "Revisar os atributos e as referências antes de definir "
        "qual registro será utilizado. Nenhuma escolha automática foi feita."
    )

# 4. Conferir disponibilidade e proporções ausentes
proporcoes = [
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas",
]

print("\n===== DISPONIBILIDADE POR CLIENTE =====")

display(
    df_features_sem_copias
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_clientes"),
        *[
            F.count(coluna).alias(f"{coluna}_preenchidas")
            for coluna in proporcoes
        ],
    )
    .orderBy("tem_endereco_disponivel")
)

resumo_nulos = df_features_sem_copias.agg(
    *[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in proporcoes
    ]
).first().asDict()

print("\n===== NULOS NAS PROPORÇÕES =====")

for coluna, quantidade in resumo_nulos.items():
    print(f"{coluna}: {quantidade}")

print("\nDomínio e padrão do e-mail foram preservados.")
print("Tamanho do hash e idade do cadastro foram preservados.")
print("As proporções ausentes continuam nulas.")
print("A base anterior e as tabelas Silver permanecem intactas.")
print("Nenhum rótulo foi criado e nenhum dado foi gravado.")

## Dicionário dos atributos

A base df_features_sem_copias possui uma linha por cliente
nos dados analisados.

| Atributo | Como foi calculado |
|---|---|
| dominio_email | Domínio após o @, em letras minúsculas, quando o e-mail atende ao formato verificado. |
| tamanho_usuario_email | Quantidade de caracteres antes do @. |
| qtd_digitos_email | Quantidade de dígitos antes do @. |
| tamanho_hash | Quantidade de caracteres do hash armazenado. |
| idade_cadastro_dias | Diferença entre ingestão e cadastro, em dias, incluindo frações. Valores negativos foram preservados. |
| qtd_enderecos_disponiveis | Quantidade de endereços ingeridos até a referência do cliente. |
| tem_endereco_disponivel | Indica se existe pelo menos um endereço disponível na referência. |
| proporcao_logradouros_ausentes | Fração dos endereços disponíveis com logradouro nulo ou vazio após remover espaços nas extremidades. |
| proporcao_coordenadas_ausentes | Fração dos endereços disponíveis com latitude ou longitude ausente. |
| proporcao_coordenadas_invalidas | Fração dos endereços disponíveis com coordenada preenchida não numérica, NaN ou fora das faixas: latitude entre -90 e 90 e longitude entre -180 e 180. |

### Interpretação dos valores

- As proporções variam de 0 a 1.
- Sem endereço disponível, a quantidade é zero e as proporções são nulas.
- Proporção nula não significa logradouro ou coordenada inválida.
- Coordenadas dentro das faixas não comprovam a localização correta.
- Os indicadores de coordenadas não apresentaram variação entre
  as referências com endereço na EDA; sua inclusão no modelo
  ainda será avaliada.

### Rastreabilidade

id_cliente, bronze_ingested_at e bronze_source_file
identificam os registros e sua origem. Não são entradas preditivas.

### Situação da preparação

Domínio, padrão do e-mail, tamanho do hash e idade do cadastro
foram preservados. Os atributos de CEP foram substituídos
pelos atributos de logradouro, com análise adicional de coordenadas.

A Silver não foi alterada.

Antes do treinamento, ainda precisamos conferir o critério
do rótulo experimental e definir o tratamento dos valores ausentes.
A codificação do domínio e qualquer transformação que aprenda
com os dados deverão ser ajustadas somente na base de treino.

In [0]:
# ============================================================
# CONFERÊNCIA DO CRITÉRIO DE DATAS
# Não cria nem modifica o rótulo.
# ============================================================

from pyspark.sql import functions as F

necessarias = [
    "id_cliente",
    "dt_cadastro",
    "bronze_ingested_at",
    "regra_05_falhou",
]

faltantes = [
    coluna
    for coluna in necessarias
    if coluna not in df_clientes_ml.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes para conferir as datas: {faltantes}"
    )

# Uma linha por combinação dos campos analisados.
# As contagens abaixo são de registros distintos nesses campos.
df_conferencia_datas = (
    df_clientes_ml
    .select(*necessarias)
    .distinct()
    .withColumn(
        "cadastro_apos_ingestao",
        F.col("dt_cadastro") > F.col("bronze_ingested_at")
    )
)

print("===== COMPARAÇÃO DOS CRITÉRIOS DE DATAS =====")

display(
    df_conferencia_datas
    .groupBy(
        "cadastro_apos_ingestao",
        "regra_05_falhou",
    )
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_cliente").alias(
            "qtd_clientes_distintos"
        ),
    )
    .orderBy(
        "cadastro_apos_ingestao",
        "regra_05_falhou",
    )
)

print("cadastro_apos_ingestao compara cadastro com ingestão.")
print("regra_05_falhou é a flag armazenada na Silver.")
print("Comparação nula indica ausência de uma das datas.")
print("Critérios diferentes podem produzir resultados diferentes.")
print("Nenhum atributo, rótulo ou dado de origem foi alterado.")

In [0]:
# ============================================================
# RÓTULO EXPERIMENTAL V2 — REGRAS ARMAZENADAS NA SILVER
#
# 1 = sinalizado por pelo menos uma das regras selecionadas.
# 0 = não sinalizado por essas regras.
# Não representa confirmação de fraude.
# ============================================================

from pyspark.sql import functions as F

chaves = ["id_cliente", "bronze_ingested_at"]

flags_rotulo = [
    "regra_04_falhou",
    "regra_05_falhou",
    "regra_09_falhou",
]

# 1. Validar as colunas e os tipos das flags
for nome, df, necessarias in [
    ("Silver de clientes", df_clientes_ml, chaves + flags_rotulo),
    ("base de atributos", df_features_sem_copias, chaves),
]:
    faltantes = [
        coluna for coluna in necessarias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

tipos_silver = dict(df_clientes_ml.dtypes)

for coluna in flags_rotulo:
    if tipos_silver[coluna] != "boolean":
        raise ValueError(
            f"{coluna} deve ser boolean, "
            f"mas está como {tipos_silver[coluna]}."
        )

colunas_novas = flags_rotulo + [
    "_flags_encontradas",
    "motivo_hash",
    "motivo_data_silver",
    "motivo_dominio",
    "rotulo_revisao_v2",
]

conflitos = [
    coluna for coluna in colunas_novas
    if coluna in df_features_sem_copias.columns
]

if conflitos:
    raise ValueError(
        f"A base de entrada já contém estas colunas: {conflitos}"
    )

# 2. Preparar as flags por cliente e instante
df_flags_rotulo_v2 = (
    df_clientes_ml
    .select(*chaves, *flags_rotulo)
    .distinct()
)

condicao_nula = F.lit(False)

for coluna in chaves + flags_rotulo:
    condicao_nula = condicao_nula | F.col(coluna).isNull()

if df_flags_rotulo_v2.filter(condicao_nula).limit(1).count() > 0:
    raise ValueError(
        "Existem chaves ou flags nulas na Silver. "
        "Revisar antes de criar o rótulo."
    )

# Interromper se a mesma referência possuir flags diferentes.
flags_conflitantes = (
    df_flags_rotulo_v2
    .groupBy(*chaves)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if flags_conflitantes > 0:
    raise ValueError(
        "Uma mesma referência possui combinações diferentes "
        "de flags na Silver. Revisar antes da junção."
    )

# 3. Relacionar as flags com a base de atributos
df_base_com_flags_v2 = (
    df_features_sem_copias
    .join(
        df_flags_rotulo_v2.withColumn(
            "_flags_encontradas", F.lit(True)
        ),
        on=chaves,
        how="left",
    )
)

total_antes = df_features_sem_copias.count()

validacao = df_base_com_flags_v2.agg(
    F.count("*").alias("total_depois"),
    F.count(
        F.when(F.col("_flags_encontradas").isNull(), 1)
    ).alias("sem_flags"),
).first()

if total_antes == 0:
    raise ValueError("A base de atributos está vazia.")

if validacao["total_depois"] != total_antes:
    raise ValueError("A quantidade de linhas mudou na junção.")

if validacao["sem_flags"] > 0:
    raise ValueError(
        "Existem registros sem correspondência nas flags da Silver."
    )

# 4. Criar os motivos e o novo rótulo
# A idade negativa continua como atributo.
# Ela não substitui a regra 05 nesta versão do rótulo.
df_base_rotulada_v2 = (
    df_base_com_flags_v2
    .withColumn("motivo_hash", F.col("regra_04_falhou"))
    .withColumn("motivo_data_silver", F.col("regra_05_falhou"))
    .withColumn("motivo_dominio", F.col("regra_09_falhou"))
    .withColumn(
        "rotulo_revisao_v2",
        (
            F.col("motivo_hash")
            | F.col("motivo_data_silver")
            | F.col("motivo_dominio")
        ).cast("int")
    )
    .drop("_flags_encontradas")
)

# 5. Exibir a distribuição
print("===== RÓTULO EXPERIMENTAL V2 =====")
print("Registros preservados:", validacao["total_depois"])
print("Critério: regra 04 OU regra 05 OU regra 09 da Silver.")

display(
    df_base_rotulada_v2
    .groupBy("rotulo_revisao_v2")
    .agg(F.count("*").alias("qtd_clientes"))
    .withColumn(
        "percentual",
        F.round(
            F.col("qtd_clientes") * 100.0 / F.lit(total_antes),
            2
        )
    )
    .orderBy("rotulo_revisao_v2")
)

print("\n===== COMBINAÇÕES DOS MOTIVOS =====")

display(
    df_base_rotulada_v2
    .groupBy(
        "motivo_hash",
        "motivo_data_silver",
        "motivo_dominio",
        "rotulo_revisao_v2",
    )
    .agg(F.count("*").alias("qtd_clientes"))
    .orderBy(F.desc("qtd_clientes"))
)

print("\nOs atributos de e-mail, hash, idade e endereços foram preservados.")
print("As flags e os motivos são destinados à auditoria do rótulo.")
print("Flags, motivos e rótulo não devem entrar como preditores.")
print("Rótulo 0 não comprova que o cadastro seja legítimo.")
print("As métricas futuras medirão concordância com essas regras.")
print("A versão anterior do rótulo não foi sobrescrita.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# BASE CANDIDATA AO MODELO
# ============================================================

from pyspark.sql import functions as F

# Identificação e rastreabilidade: não entram como preditores.
colunas_identificacao = [
    "id_cliente",
    "bronze_ingested_at",
    "bronze_source_file",
]

# Atributo categórico: será codificado na etapa de modelagem.
features_categoricas = [
    "dominio_email",
]

features_numericas = [
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
    "qtd_enderecos_disponiveis",
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas",
]

features_booleanas = [
    "tem_endereco_disponivel",
]

features_candidatas = (
    features_categoricas
    + features_numericas
    + features_booleanas
)

coluna_alvo = "rotulo_revisao_v2"

# 1. Validar as colunas necessárias
colunas_base = (
    colunas_identificacao
    + features_candidatas
    + [coluna_alvo]
)

faltantes = [
    coluna
    for coluna in colunas_base
    if coluna not in df_base_rotulada_v2.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes na base rotulada: {faltantes}"
    )

# 2. Selecionar explicitamente os campos
# Flags e motivos permanecem em df_base_rotulada_v2 para auditoria.
df_base_modelo = df_base_rotulada_v2.select(*colunas_base)

# 3. Validar os valores do rótulo
rotulos_invalidos = (
    df_base_modelo
    .filter(
        F.col(coluna_alvo).isNull()
        | (~F.col(coluna_alvo).isin(0, 1))
    )
    .limit(1)
    .count()
)

if rotulos_invalidos > 0:
    raise ValueError("Existem rótulos nulos ou diferentes de 0 e 1.")

# 4. Resumir a base e conferir a rastreabilidade dos lotes
resumo = df_base_modelo.agg(
    F.count("*").alias("registros"),
    F.countDistinct("id_cliente").alias("clientes_distintos"),
    F.countDistinct("bronze_source_file").alias(
        "arquivos_origem_distintos"
    ),
    F.count(
        F.when(
            F.col("bronze_source_file").isNull()
            | (F.trim(F.col("bronze_source_file")) == ""),
            1
        )
    ).alias("linhas_sem_arquivo_origem"),
    F.min("bronze_ingested_at").alias("primeira_ingestao"),
    F.max("bronze_ingested_at").alias("ultima_ingestao"),
).first().asDict()

print("===== BASE CANDIDATA AO MODELO =====")

for indicador, valor in resumo.items():
    print(f"{indicador}: {valor}")

print("\n===== ATRIBUTOS CANDIDATOS =====")

for coluna in features_candidatas:
    print("-", coluna)

print("\nAlvo:", coluna_alvo)

# 5. Conferir ausência e variação dos atributos
# countDistinct não inclui valores nulos.
expressoes = []

for coluna in features_candidatas:
    expressoes.extend([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(f"{coluna}__nulos"),
        F.countDistinct(coluna).alias(f"{coluna}__distintos"),
    ])

perfil = df_base_modelo.agg(*expressoes).first().asDict()

print("\n===== AUSÊNCIA E VARIAÇÃO =====")

for coluna in features_candidatas:
    print(
        f"{coluna}: "
        f"{perfil[coluna + '__nulos']} nulos; "
        f"{perfil[coluna + '__distintos']} valores distintos não nulos"
    )

print("\nAs proporções ausentes continuam nulas.")
print("A lista contém candidatos; a seleção final ainda será definida.")
print("Identificadores, flags e motivos não são preditores.")
print("Nenhuma codificação ou imputação foi ajustada.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
print("===== RASTREABILIDADE DOS LOTES =====")

for indicador in [
    "registros",
    "clientes_distintos",
    "arquivos_origem_distintos",
    "linhas_sem_arquivo_origem",
    "primeira_ingestao",
    "ultima_ingestao",
]:
    print(f"{indicador}: {resumo[indicador]}")

In [0]:
# ============================================================
# ORGANIZAÇÃO DOS LOTES PARA A FUTURA DIVISÃO TEMPORAL
# Nenhuma separação ou treinamento é realizado nesta célula.
# ============================================================

from pyspark.sql import functions as F

df_resumo_lotes = (
    df_base_modelo
    .groupBy("bronze_source_file")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.min("bronze_ingested_at").alias("primeira_ingestao"),
        F.max("bronze_ingested_at").alias("ultima_ingestao"),
        F.countDistinct("bronze_ingested_at").alias(
            "qtd_instantes_ingestao"
        ),
        F.count(
            F.when(F.col("bronze_ingested_at").isNull(), 1)
        ).alias("ingestoes_nulas"),
    )
)

resumo_lotes = df_resumo_lotes.agg(
    F.count("*").alias("total_arquivos"),
    F.sum("qtd_clientes").alias("total_clientes"),
    F.count(
        F.when(F.col("qtd_instantes_ingestao") > 1, 1)
    ).alias("arquivos_com_multiplos_instantes"),
    F.sum("ingestoes_nulas").alias("ingestoes_nulas"),
    F.min("qtd_clientes").alias("menor_lote"),
    F.max("qtd_clientes").alias("maior_lote"),
).first().asDict()

print("===== ORGANIZAÇÃO DOS LOTES =====")

for indicador, valor in resumo_lotes.items():
    print(f"{indicador}: {valor}")

print("\n===== DISTRIBUIÇÃO POR DIA DE INGESTÃO =====")

display(
    df_base_modelo
    .groupBy(
        F.to_date("bronze_ingested_at").alias("dia_ingestao")
    )
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.countDistinct("bronze_source_file").alias("qtd_arquivos"),
    )
    .orderBy("dia_ingestao")
)

print("\nUm arquivo pode aparecer em mais de um dia se tiver várias ingestões.")
print("Nenhum arquivo foi dividido entre treino e avaliação.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# SEPARAÇÃO TEMPORAL — TREINO E AVALIAÇÃO
# Corte definido pelo período, sem consultar desempenho.
# ============================================================

from pyspark.sql import functions as F

data_corte = "2026-10-16"
alvo = "rotulo_revisao_v2"

# 1. Separar pela data de ingestão
dia_ingestao = F.to_date("bronze_ingested_at")
corte = F.lit(data_corte).cast("date")

df_treino = df_base_modelo.filter(dia_ingestao < corte)
df_avaliacao = df_base_modelo.filter(dia_ingestao >= corte)

# 2. Conferir a preservação dos registros
total_base = df_base_modelo.count()
total_treino = df_treino.count()
total_avaliacao = df_avaliacao.count()

if total_treino == 0 or total_avaliacao == 0:
    raise ValueError(
        "Uma das bases ficou vazia. Revise o corte temporal."
    )

if total_treino + total_avaliacao != total_base:
    raise ValueError(
        "A divisão não preservou todas as linhas. "
        "Confira as datas de ingestão."
    )

# 3. Garantir que nenhum arquivo apareça nas duas bases
arquivos_compartilhados = (
    df_treino.select("bronze_source_file").distinct()
    .join(
        df_avaliacao.select("bronze_source_file").distinct(),
        on="bronze_source_file",
        how="inner",
    )
    .count()
)

if arquivos_compartilhados > 0:
    raise ValueError(
        "Existem arquivos compartilhados entre treino e avaliação."
    )

# 4. Garantir que nenhum cliente apareça nas duas bases
clientes_compartilhados = (
    df_treino.select("id_cliente").distinct()
    .join(
        df_avaliacao.select("id_cliente").distinct(),
        on="id_cliente",
        how="inner",
    )
    .count()
)

if clientes_compartilhados > 0:
    raise ValueError(
        "Existem clientes compartilhados entre treino e avaliação."
    )

# 5. Resumir as duas bases
df_divisao_resumo = (
    df_treino.select(
        "bronze_source_file", "bronze_ingested_at", alvo
    )
    .withColumn("base", F.lit("TREINO"))
    .unionByName(
        df_avaliacao.select(
            "bronze_source_file", "bronze_ingested_at", alvo
        )
        .withColumn("base", F.lit("AVALIACAO"))
    )
)

print("===== SEPARAÇÃO TEMPORAL =====")
print("Data de corte:", data_corte)
print("Arquivos compartilhados:", arquivos_compartilhados)
print("Clientes compartilhados:", clientes_compartilhados)

display(
    df_divisao_resumo
    .groupBy("base")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.countDistinct("bronze_source_file").alias("qtd_arquivos"),
        F.min("bronze_ingested_at").alias("primeira_ingestao"),
        F.max("bronze_ingested_at").alias("ultima_ingestao"),
    )
    .orderBy("base")
)

print("\n===== CLASSES POR BASE =====")

display(
    df_divisao_resumo
    .groupBy("base", alvo)
    .agg(F.count("*").alias("qtd_clientes"))
    .orderBy("base", alvo)
)

classes_treino = df_treino.select(alvo).distinct().count()
classes_avaliacao = df_avaliacao.select(alvo).distinct().count()

if classes_treino != 2:
    raise ValueError(
        "O treino não contém as duas classes. "
        "Não iniciar o treinamento."
    )

if classes_avaliacao != 2:
    print(
        "ATENÇÃO: a avaliação contém apenas uma classe. "
        "Isso limita a interpretação das métricas."
    )

print("\nNenhuma codificação ou imputação foi ajustada.")
print("A avaliação deve permanecer reservada para medir o modelo.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# PREPARAÇÃO DAS ENTRADAS DO PRIMEIRO MODELO
# ============================================================

from pyspark.sql import functions as F

features_categoricas = [
    "dominio_email",
]

features_numericas_modelo = [
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
]

features_modelo = (
    features_categoricas + features_numericas_modelo
)

alvo_modelo = "rotulo_revisao_v2"

colunas_rastreabilidade = [
    "id_cliente",
    "bronze_ingested_at",
    "bronze_source_file",
]


def preparar_entradas(df):
    """Aplica regras fixas, sem aprender com treino ou avaliação."""

    necessarias = (
        colunas_rastreabilidade
        + features_modelo
        + [alvo_modelo]
    )

    faltantes = [
        coluna for coluna in necessarias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}")

    # A ausência da proporção deve corresponder à falta de endereço.
    disponivel = F.col("tem_endereco_disponivel")
    proporcao = F.col("proporcao_logradouros_ausentes")
    quantidade = F.col("qtd_enderecos_disponiveis")

    inconsistente = (
        disponivel.isNull()
        | quantidade.isNull()
        | (quantidade < 0)
        | (disponivel != (quantidade > 0))
        | (
            disponivel
            & (
                proporcao.isNull()
                | F.isnan(proporcao)
                | (~proporcao.between(0.0, 1.0))
            )
        )
        | (
            (~disponivel)
            & proporcao.isNotNull()
        )
    )

    if df.filter(inconsistente).limit(1).count() > 0:
        raise ValueError(
            "Disponibilidade, quantidade e proporção de "
            "logradouros estão inconsistentes."
        )

    return (
        df
        .withColumn(
            "tem_endereco_disponivel",
            F.col("tem_endereco_disponivel").cast("int")
        )
        .withColumn(
            "proporcao_logradouros_ausentes",
            F.when(
                F.col("tem_endereco_disponivel") == 0,
                F.lit(-1.0)
            ).otherwise(
                F.col("proporcao_logradouros_ausentes")
            )
        )
        .select(*necessarias)
    )


# 1. Aplicar a mesma transformação fixa às duas bases
df_treino_preparado = preparar_entradas(df_treino)
df_avaliacao_preparada = preparar_entradas(df_avaliacao)

# 2. Conferir nulos e NaN nas entradas
for nome, df in [
    ("TREINO", df_treino_preparado),
    ("AVALIACAO", df_avaliacao_preparada),
]:
    entrada_invalida = F.lit(False)

    for coluna in features_modelo:
        entrada_invalida = (
            entrada_invalida | F.col(coluna).isNull()
        )

    for coluna in features_numericas_modelo:
        entrada_invalida = (
            entrada_invalida
            | F.isnan(F.col(coluna).cast("double"))
        )

    entrada_invalida = (
        entrada_invalida
        | (F.trim(F.col("dominio_email")) == "")
    )

    if df.filter(entrada_invalida).limit(1).count() > 0:
        raise ValueError(
            f"{nome}: existem entradas nulas, NaN "
            "ou domínios vazios. Revisar antes de continuar."
        )

    print(f"\n===== {nome} =====")
    print("Registros:", df.count())
    print(
        "Sem endereço disponível:",
        df.filter(
            F.col("tem_endereco_disponivel") == 0
        ).count()
    )

print("\n===== ENTRADAS DO PRIMEIRO MODELO =====")

for coluna in features_modelo:
    print("-", coluna)

print("\nAlvo:", alvo_modelo)
print("Proporção -1 significa ausência de endereço disponível.")
print("Proporção 0 significa nenhum logradouro ausente nos endereços avaliados.")
print("O domínio ainda será codificado usando somente o treino.")
print("As coordenadas permanecem nas bases anteriores para análise.")
print("As colunas de rastreabilidade não pertencem à lista de entradas.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# PREPARAÇÃO DAS ENTRADAS DO PRIMEIRO MODELO
# ============================================================

from pyspark.sql import functions as F

features_categoricas = [
    "dominio_email",
]

features_numericas_modelo = [
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
]

features_modelo = (
    features_categoricas + features_numericas_modelo
)

alvo_modelo = "rotulo_revisao_v2"

colunas_rastreabilidade = [
    "id_cliente",
    "bronze_ingested_at",
    "bronze_source_file",
]


def preparar_entradas(df):
    """Aplica regras fixas, sem aprender com treino ou avaliação."""

    necessarias = (
        colunas_rastreabilidade
        + features_modelo
        + [alvo_modelo]
    )

    faltantes = [
        coluna for coluna in necessarias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}")

    # A ausência da proporção deve corresponder à falta de endereço.
    disponivel = F.col("tem_endereco_disponivel")
    proporcao = F.col("proporcao_logradouros_ausentes")
    quantidade = F.col("qtd_enderecos_disponiveis")

    inconsistente = (
        disponivel.isNull()
        | quantidade.isNull()
        | (quantidade < 0)
        | (disponivel != (quantidade > 0))
        | (
            disponivel
            & (
                proporcao.isNull()
                | F.isnan(proporcao)
                | (~proporcao.between(0.0, 1.0))
            )
        )
        | (
            (~disponivel)
            & proporcao.isNotNull()
        )
    )

    if df.filter(inconsistente).limit(1).count() > 0:
        raise ValueError(
            "Disponibilidade, quantidade e proporção de "
            "logradouros estão inconsistentes."
        )

    return (
        df
        .withColumn(
            "tem_endereco_disponivel",
            F.col("tem_endereco_disponivel").cast("int")
        )
        .withColumn(
            "proporcao_logradouros_ausentes",
            F.when(
                F.col("tem_endereco_disponivel") == 0,
                F.lit(-1.0)
            ).otherwise(
                F.col("proporcao_logradouros_ausentes")
            )
        )
        .select(*necessarias)
    )


# 1. Aplicar a mesma transformação fixa às duas bases
df_treino_preparado = preparar_entradas(df_treino)
df_avaliacao_preparada = preparar_entradas(df_avaliacao)

# 2. Conferir nulos e NaN nas entradas
for nome, df in [
    ("TREINO", df_treino_preparado),
    ("AVALIACAO", df_avaliacao_preparada),
]:
    entrada_invalida = F.lit(False)

    for coluna in features_modelo:
        entrada_invalida = (
            entrada_invalida | F.col(coluna).isNull()
        )

    for coluna in features_numericas_modelo:
        entrada_invalida = (
            entrada_invalida
            | F.isnan(F.col(coluna).cast("double"))
        )

    entrada_invalida = (
        entrada_invalida
        | (F.trim(F.col("dominio_email")) == "")
    )

    if df.filter(entrada_invalida).limit(1).count() > 0:
        raise ValueError(
            f"{nome}: existem entradas nulas, NaN "
            "ou domínios vazios. Revisar antes de continuar."
        )

    print(f"\n===== {nome} =====")
    print("Registros:", df.count())
    print(
        "Sem endereço disponível:",
        df.filter(
            F.col("tem_endereco_disponivel") == 0
        ).count()
    )

print("\n===== ENTRADAS DO PRIMEIRO MODELO =====")

for coluna in features_modelo:
    print("-", coluna)

print("\nAlvo:", alvo_modelo)
print("Proporção -1 significa ausência de endereço disponível.")
print("Proporção 0 significa nenhum logradouro ausente nos endereços avaliados.")
print("O domínio ainda será codificado usando somente o treino.")
print("As coordenadas permanecem nas bases anteriores para análise.")
print("As colunas de rastreabilidade não pertencem à lista de entradas.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# CODIFICAÇÃO DO DOMÍNIO — APRENDIDA SOMENTE NO TREINO
# ============================================================

from pyspark.sql import functions as F

# 1. Obter os domínios exclusivamente do treino
# Limite para evitar coletar uma lista inesperadamente grande.
linhas_dominios = (
    df_treino_preparado
    .select("dominio_email")
    .distinct()
    .orderBy("dominio_email")
    .limit(1001)
    .collect()
)

if len(linhas_dominios) > 1000:
    raise ValueError(
        "Mais de 1.000 domínios no treino. "
        "Revisar a estratégia de codificação."
    )

dominios_treino = [
    linha["dominio_email"]
    for linha in linhas_dominios
]

if not dominios_treino:
    raise ValueError("Nenhum domínio encontrado no treino.")

if any(
    dominio is None or not dominio.strip()
    for dominio in dominios_treino
):
    raise ValueError("Existem domínios nulos ou vazios no treino.")

# Nomes simples e mapeamento explícito para documentar as colunas.
mapa_dominios = [
    (dominio, f"dominio_email_{indice:02d}")
    for indice, dominio in enumerate(dominios_treino, start=1)
]

colunas_dominios = [
    coluna for dominio, coluna in mapa_dominios
]

coluna_desconhecido = "dominio_desconhecido"


# 2. Aplicar o mesmo mapeamento às duas bases
def codificar_dominio(df):
    novas_colunas = colunas_dominios + [coluna_desconhecido]

    conflitos = [
        coluna for coluna in novas_colunas
        if coluna in df.columns
    ]

    if conflitos:
        raise ValueError(
            f"A base já contém colunas de codificação: {conflitos}"
        )

    indicadores = [
        F.when(
            F.col("dominio_email") == F.lit(dominio),
            F.lit(1)
        ).otherwise(F.lit(0)).alias(coluna)
        for dominio, coluna in mapa_dominios
    ]

    desconhecido = (
        F.when(
            F.col("dominio_email").isin(*dominios_treino),
            F.lit(0)
        )
        .otherwise(F.lit(1))
        .alias(coluna_desconhecido)
    )

    return df.select("*", *indicadores, desconhecido)


df_treino_codificado = codificar_dominio(df_treino_preparado)
df_avaliacao_codificada = codificar_dominio(df_avaliacao_preparada)

# 3. Definir explicitamente as entradas numéricas finais
features_finais = (
    features_numericas_modelo
    + colunas_dominios
    + [coluna_desconhecido]
)

# O domínio original permanece para rastreabilidade,
# mas não está em features_finais.
print("===== MAPEAMENTO DOS DOMÍNIOS =====")

for dominio, coluna in mapa_dominios:
    print(f"{coluna}: {dominio}")

print(f"{coluna_desconhecido}: domínio não encontrado no treino")

# 4. Validar a codificação
# Cada linha deve ter exatamente um indicador de domínio ativo.
soma_indicadores = F.lit(0)

for coluna in colunas_dominios + [coluna_desconhecido]:
    soma_indicadores = soma_indicadores + F.col(coluna)

for nome, df in [
    ("TREINO", df_treino_codificado),
    ("AVALIACAO", df_avaliacao_codificada),
]:
    if df.filter(soma_indicadores != 1).limit(1).count() > 0:
        raise ValueError(
            f"{nome}: codificação inconsistente dos domínios."
        )

    resumo_codificacao = df.agg(
        F.count("*").alias("registros"),
        F.count(
            F.when(F.col(coluna_desconhecido) == 1, 1)
        ).alias("registros_com_dominio_desconhecido"),
    ).first()

    print(f"\n===== {nome} =====")
    print("Registros:", resumo_codificacao["registros"])
    print(
        "Registros com domínio desconhecido:",
        resumo_codificacao["registros_com_dominio_desconhecido"]
    )

print("\n===== ENTRADAS NUMÉRICAS FINAIS =====")

for coluna in features_finais:
    print("-", coluna)

print("\nQuantidade de entradas:", len(features_finais))
print("Alvo:", alvo_modelo)
print("Domínios aprendidos exclusivamente no treino.")
print("Domínio desconhecido não significa fraude ou falha de cadastro.")
print("IDs, flags, motivos e rótulo não fazem parte das entradas.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")